# 📓 05 — MuJoCo SimulationCreate physics simulations, add robots and objects, render cameras, and controleverything programmatically or through a Strands Agent.**Requires**: `pip install "strands-robots[sim-mujoco]"`

In [ ]:
from strands_robots.simulation import create_simulation, Simulation, list_backendsprint(f"Available backends: {list_backends()}")

## Creating a World

In [ ]:
# Create simulation (default: MuJoCo)sim = create_simulation("mujoco")# Create world with custom physicsresult = sim.create_world(    timestep=0.002,       # 500Hz physics    gravity=[0, 0, -9.81],    ground_plane=True,)print(result["content"][0]["text"])

## Adding RobotsRobots are resolved from the model registry — supports Menagerie, custom URDFs, and more.

In [ ]:
# Add a robot by name (auto-resolves MJCF from registry)result = sim.add_robot(    name="so100",    data_config="so100",    position=[0, 0, 0],)print(result["content"][0]["text"])

## Adding Objects

In [ ]:
# Add various primitive shapessim.add_object("red_cube", shape="box", position=[0.3, 0, 0.05],               size=[0.03, 0.03, 0.03], color=[1, 0, 0, 1], mass=0.05)sim.add_object("blue_sphere", shape="sphere", position=[0.2, 0.1, 0.05],               size=[0.02], color=[0, 0, 1, 1], mass=0.03)sim.add_object("green_cylinder", shape="cylinder", position=[0.25, -0.1, 0.05],               size=[0.02, 0.04], color=[0, 1, 0, 1], mass=0.04)# Static objects (immovable)sim.add_object("table", shape="box", position=[0.3, 0, -0.02],               size=[0.3, 0.3, 0.02], color=[0.6, 0.4, 0.2, 1], is_static=True)print("Objects added ✅")

## Physics Stepping & Observation

In [ ]:
# Step the simulationsim.step(n_steps=100)# Get observation (joint positions + camera images)obs = sim.get_observation("so100")joint_keys = [k for k in obs if not hasattr(obs[k], 'shape') or obs[k].ndim < 2]image_keys = [k for k in obs if hasattr(obs[k], 'shape') and obs[k].ndim >= 2]print(f"Joint states: {joint_keys}")for k in joint_keys:    print(f"  {k}: {obs[k]:.4f}")print(f"\nCamera images: {image_keys}")for k in image_keys:    print(f"  {k}: shape={obs[k].shape}, dtype={obs[k].dtype}")

## Rendering

In [ ]:
# Render the default cameraresult = sim.render(camera_name="default", width=640, height=480)if "image" in result:    img = result["image"]    print(f"Rendered image: {img.shape} {img.dtype}")    # Display in notebook (if matplotlib available)    try:        import matplotlib.pyplot as plt        plt.figure(figsize=(8, 6))        plt.imshow(img)        plt.title("MuJoCo Simulation")        plt.axis("off")        plt.show()    except ImportError:        print("Install matplotlib for inline display")else:    print("Rendering not available (headless without EGL/OSMesa)")

## Sending Actions

In [ ]:
import numpy as np# Apply joint position controlaction = {}for key in joint_keys:    action[key] = float(np.sin(0.5))  # simple position targetsim.send_action(action, robot_name="so100")# Step to applyfor _ in range(50):    sim.step(n_steps=1)# Check new positionsnew_obs = sim.get_observation("so100")for k in joint_keys:    print(f"  {k}: target={action[k]:.4f} → actual={new_obs[k]:.4f}")

## Advanced Physics (PhysicsMixin)The simulation exposes deep MuJoCo C API features.

In [ ]:
# Save/restore state checkpointssim.save_state("before_action")# Do some stepsfor _ in range(100):    sim.step()# Restore to saved statesim.load_state("before_action")print("State restored ✅")

In [ ]:
# Cleanupsim.destroy()print("World destroyed ✅")